# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [12]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

MODEL_online = 'gpt-4o-mini'
MODEL_local = 'llama3.2'
set_local = False
MODEL = MODEL_local if set_local else MODEL_online
print(MODEL)
# openai = OpenAI()
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama') if set_local else OpenAI()

API key looks good so far
gpt-4o-mini


In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://verizon.com")
ed.links

['https://www.verizon.com/accessibility',
 '#gnav20-header-end',
 'https://www.verizon.com/',
 'https://www.verizon.com/business/?cmp=vcgref_naveyebrow',
 'tel:+18338374966',
 'https://www.verizon.com/support/contact-us/',
 'https://www.verizon.com/support/',
 'https://www.verizon.com/stores/',
 'https://www.verizon.com/coverage-map/',
 'https://espanol.verizon.com/',
 'https://www.verizon.com/',
 'https://www.verizon.com/plans/unlimited/',
 'https://www.verizon.com/plans/unlimited/',
 'https://www.verizon.com/home/internet/',
 'https://www.verizon.com/home/internet/',
 'https://www.verizon.com/shop/',
 'https://www.verizon.com/shop/',
 'https://www.verizon.com/smartphones/',
 'javascript:void(0)',
 'https://www.verizon.com/smartphones/',
 'https://www.verizon.com/smartphones/certified-pre-owned/',
 'javascript:void(0)',
 'https://www.verizon.com/smartphones/apple-iphone-16-pro/',
 'https://www.verizon.com/smartphones/apple-iphone-16/',
 'https://www.verizon.com/smartphones/samsung-gal

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}



In [7]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [8]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://verizon.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://www.verizon.com/accessibility
#gnav20-header-end
https://www.verizon.com/
https://www.verizon.com/business/?cmp=vcgref_naveyebrow
tel:+18338374966
https://www.verizon.com/support/contact-us/
https://www.verizon.com/support/
https://www.verizon.com/stores/
https://www.verizon.com/coverage-map/
https://espanol.verizon.com/
https://www.verizon.com/
https://www.verizon.com/plans/unlimited/
https://www.verizon.com/plans/unlimited/
https://www.verizon.com/home/internet/
https://www.verizon.com/home/internet/
https://www.verizon.com/shop/
https://www.verizon.com/shop/
https://www.verizon.com/smartphones/
javascript:void(0)
https://www.verizon.com/smartphones/
https://www.verizon.com/smartphones/ce

In [9]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

In [10]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://verizon.com")
huggingface.links

['https://www.verizon.com/accessibility',
 '#gnav20-header-end',
 'https://www.verizon.com/',
 'https://www.verizon.com/business/?cmp=vcgref_naveyebrow',
 'tel:+18338374966',
 'https://www.verizon.com/support/contact-us/',
 'https://www.verizon.com/support/',
 'https://www.verizon.com/stores/',
 'https://www.verizon.com/coverage-map/',
 'https://espanol.verizon.com/',
 'https://www.verizon.com/',
 'https://www.verizon.com/plans/unlimited/',
 'https://www.verizon.com/plans/unlimited/',
 'https://www.verizon.com/home/internet/',
 'https://www.verizon.com/home/internet/',
 'https://www.verizon.com/shop/',
 'https://www.verizon.com/shop/',
 'https://www.verizon.com/smartphones/',
 'javascript:void(0)',
 'https://www.verizon.com/smartphones/',
 'https://www.verizon.com/smartphones/certified-pre-owned/',
 'javascript:void(0)',
 'https://www.verizon.com/smartphones/apple-iphone-16-pro/',
 'https://www.verizon.com/smartphones/apple-iphone-16/',
 'https://www.verizon.com/smartphones/samsung-gal

get_links("https://huggingface.co")

In [13]:
get_links("https://verizon.com")

{'links': [{'type': 'about page', 'url': 'https://www.verizon.com/about'},
  {'type': 'careers page', 'url': 'https://mycareer.verizon.com/'},
  {'type': 'company news', 'url': 'https://www.verizon.com/about/news'},
  {'type': 'corporate responsibility',
   'url': 'https://www.verizon.com/about/responsibility'},
  {'type': 'company innovation',
   'url': 'https://www.verizon.com/about/our-company/innovation-labs'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [14]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

print(get_all_details("https://huggingface.co"))

In [15]:
print(get_all_details("https://verizon.com"))

Found links: {'links': [{'type': 'about page', 'url': 'https://www.verizon.com/about'}, {'type': 'company page', 'url': 'https://www.verizon.com/about/our-company'}, {'type': 'careers page', 'url': 'https://mycareer.verizon.com/'}, {'type': 'news page', 'url': 'https://www.verizon.com/about/news'}, {'type': 'corporate responsibility page', 'url': 'https://www.verizon.com/about/responsibility'}, {'type': 'innovation labs page', 'url': 'https://www.verizon.com/about/our-company/innovation-labs'}]}
Landing page:
Webpage Title:
Verizon: Wireless, Internet, TV and Phone Services | Official Site
Webpage Contents:
Accessibility Resource Center
Skip to main content
Personal
Business
1-833-VERIZON
Contact us
Support
Stores
Coverage map
Español
Mobile
Mobile
Mobile
Close
Home Internet
Home Internet
Home Internet
Close
Shop
Shop
Shop
Shop
Shop all
Devices
Devices
Devices
Smartphones
Certified pre-owned phones
Featured smartphones
Featured smartphones
Featured smartphones
Apple iPhone 16 Pro
Apple

In [26]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("Verizon", "https://verizon.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.verizon.com/about'}, {'type': 'company page', 'url': 'https://www.verizon.com/about/our-company'}, {'type': 'careers page', 'url': 'https://mycareer.verizon.com/'}, {'type': 'news page', 'url': 'https://www.verizon.com/about/news'}, {'type': 'responsibility page', 'url': 'https://www.verizon.com/about/responsibility'}]}


'You are looking at a company called: Verizon\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nVerizon: Wireless, Internet, TV and Phone Services | Official Site\nWebpage Contents:\nAccessibility Resource Center\nSkip to main content\nPersonal\nBusiness\n1-833-VERIZON\nContact us\nSupport\nStores\nCoverage map\nEspañol\nMobile\nMobile\nMobile\nClose\nHome Internet\nHome Internet\nHome Internet\nClose\nShop\nShop\nShop\nShop\nShop all\nDevices\nDevices\nDevices\nSmartphones\nCertified pre-owned phones\nFeatured smartphones\nFeatured smartphones\nFeatured smartphones\nApple iPhone 16 Pro\nApple iPhone 16\nSamsung Galaxy S25 Ultra\nGoogle Pixel 9 Pro\nUpgrade your device\nBring your own device\nUnlocked phones\nOther phones\nTrade in your device\nTablets & laptops\nWatches\nCertified pre-owned watches\nJetpacks & hotspots\nMobile plans\nMobile plans\nMobile plans\

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("Verizon", "https://verizon.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.verizon.com/about'}, {'type': 'company page', 'url': 'https://www.verizon.com/about/our-company/open-internet'}, {'type': 'careers page', 'url': 'https://mycareer.verizon.com/'}, {'type': 'news page', 'url': 'https://www.verizon.com/about/news'}, {'type': 'responsibility page', 'url': 'https://www.verizon.com/about/responsibility'}, {'type': 'accessibility page', 'url': 'https://www.verizon.com/accessibility'}]}


# Welcome to the Wonderful World of Verizon!

### Your Connection to Everything (Well, Almost Everything)
Welcome to **Verizon**, where we make sure you’re connected to all the things you love — like Netflix, TikTok, and that incessant group chat no one can ignore. We provide wireless, internet, TV, and phone services to keep you plugged in, tuned out, and only mildly frustrated when you lose your signal. 

---

### 🌐 What Do We Offer?
- **Mobile Plans That Make You Go "Whoa!"**  
  Pick a plan as limitless as your weekend plans! Whether you’re streaming, gaming, or pretending to work, we’ve got you covered.
  
- **Home Internet**  
  Our internet connection is so reliable, you might just start to think your Wi-Fi is your only friend. No annual contracts to hold you back—just a solid connection when it counts.

- **TV & Entertainment**  
  Enjoy the NFL Sunday Ticket on us (because who doesn’t want to watch athletes tackle each other on Sunday?). Subscribe to our services, and you may just find yourself yelling at the TV in pure joy or outrage.

---

### 🌟 Company Culture (Read: Fun and Functional)
At Verizon, we believe in the right balance of seriousness and silliness. Our office vibe? Think of it like a family dinner—plenty of quirk and fun, but we still know how to get down to business when it comes to serving our customers. 

👩‍💼 **Growth and Development:** We’re all about boosting career skills that lead to personal growth and a workplace where no idea, no matter how outlandish, goes unconsidered.

🧑‍🤝‍🧑 **Team Spirit:** We play hard and work harder! Join an environment that encourages collaboration that’s wirelessly greater than the sum of its parts.

---

### 🎉 Who Loves Us? 
Everyone! From families picking their plans like kids in a candy store to businesses trying to make sense of the digital world, we cater to all.  
👨‍👩‍👧 **Families:** Save big when everyone can choose their own plan!" Smart thinking, or we just like to keep family squabbles to a minimum. 

---

### 💼 Careers
Dreaming of a job where the Wi-Fi is as strong as the coffee? At Verizon, we’re looking for dynamic individuals who aren't afraid to innovate, collaborate, and occasionally debate whether pineapple belongs on pizza.

**Perks include:**
- Competitive salaries that won’t make you want to throw your phone out of frustration.
- Opportunities to contribute to projects that genuinely *matter*.
- A comprehensive employee discount on mobile plans (because we totally believe in practicing what we preach).

---

### 📞 Get Connected!
Phone us at **1-833-VERIZON** or check out our website to explore more about our services, deals, and career opportunities. 
Because who doesn’t want to be on a first-name basis with their wireless provider?

--- 

**Verizon**: Keeping you connected, entertained, and maybe just a little bit too connected to your devices. But hey, that's what we're here for! 😄

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("Verizon", "https://verizon.com")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.verizon.com/about'}, {'type': 'company page', 'url': 'https://www.verizon.com/about/our-company/innovation-labs'}, {'type': 'careers page', 'url': 'https://mycareer.verizon.com/'}, {'type': 'news page', 'url': 'https://www.verizon.com/about/news'}]}


# Verizon Company Brochure

## About Verizon
Verizon is a leading telecommunications provider, dedicated to delivering high-speed wireless and internet services to both personal and business customers across the United States. Serving millions with innovative technologies, Verizon offers a comprehensive suite of services, including mobile, home internet, TV, and phone solutions. With a commitment to enhancing customer experiences through reliable connections and cutting-edge devices, Verizon is shaping the future of communications.

### Our Services

- **Mobile Solutions**: 
  - Plans: Unlimited, international, and prepaid options.
  - Devices: Latest smartphones, smartwatches, tablets, and more.
  - Accessories: From phone cases to wearable tech.

- **Home Internet & TV**: 
  - High-speed internet options: Fios Home Internet, 5G Home Internet.
  - Television services: Fios TV with numerous channels and packages.
  - Bundled offers that combine wireless and home services for added savings.

### Customers
Verizon serves a diverse customer base, including individuals, families, and businesses. Our personalized approach allows families to choose custom plans while businesses receive tailored solutions to enhance productivity. With nationwide coverage and a commitment to network reliability, our customers can stay connected anytime, anywhere.

### Company Culture
At Verizon, we foster a culture of innovation, collaboration, and inclusivity. Our commitment to diversity and a supportive work environment is reflected in our core values. We believe in empowering employees to challenge the status quo, push boundaries, and deliver exceptional service. Continuous learning and professional growth are not only encouraged, but integral to our success as a leading telecommunications firm.

### Careers at Verizon
Join a forward-thinking team at Verizon! We offer a wide range of career opportunities in technology, customer service, sales, and more. Our focus on employee development means you will have access to training programs and resources to excel in your career. We’re looking for passionate individuals who are ready to shape the future of telecommunications.

- **Why Work with Us?**
  - Competitive salaries and comprehensive benefits.
  - Opportunities for advancement and professional training.
  - A culture that prioritizes health, work-life balance, and community engagement.

---

For more information, visit us at [Verizon Official Site](https://www.verizon.com) or contact us at 1-833-VERIZON. Whether you are a customer, potential investor, or future employee, Verizon is excited to connect with you!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>